In [32]:
from pathlib import Path
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import yaml
import numpy as np

In [33]:
REPO = Path.home() / "Documents" / "GitHub" / "KNdata-public"
DATA_ROOT = REPO / "caves_fulldatasets"
summary_file = DATA_ROOT / "caves_properties_summary.csv"

print("Summary file:", summary_file)
print("Exists?", summary_file.exists())

table_caves = pd.read_csv(
    summary_file,
    sep=";",
    index_col=1,
    dtype={"id_subset": str, "id_dataset": str},
)

print(type(table_caves))
print(table_caves.head())
print(table_caves.columns)

Summary file: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\caves_properties_summary.csv
Exists? True
<class 'pandas.core.frame.DataFrame'>
              id_dataset id_subset status         type_of_correction  \
short_name                                                             
GouffreDejaVu        001       000  clean              no correction   
Migovec              002       000  clean          manual correction   
Criou                003       000  clean          manual correction   
Matienzo             004       000  clean  semi-automatic correction   
Sakany               005       000  clean          manual correction   

              last_processing_date  total_length  total_depth  \
short_name                                                      
GouffreDejaVu          11 Nov 2025           150        32.16   
Migovec                11 Nov 2025         46317       968.62   
Criou                  11 Nov 2025         31680      1669.43   
Matienzo   

In [34]:
def spectral_invariants(G, tol=1e-9):
    if G.number_of_nodes() == 0:
        raise ValueError("Graph is empty.")

    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
        print("Graph was disconnected; using largest connected component.")

    nodes = list(G.nodes())
    n = len(nodes)
    m = G.number_of_edges()

    A = nx.to_numpy_array(G, nodelist=nodes, dtype=float)
    L = nx.laplacian_matrix(G, nodelist=nodes).toarray().astype(float)
    degrees = A.sum(axis=1)
    D = np.diag(degrees)
    Q = D + A

    D_inv_sqrt = np.zeros((n, n), dtype=float)
    for i in range(n):
        if degrees[i] > tol:
            D_inv_sqrt[i, i] = 1.0 / np.sqrt(degrees[i])
    Lnorm = np.eye(n) - D_inv_sqrt @ A @ D_inv_sqrt

    adj_eigs = np.sort(np.linalg.eigvalsh(A))[::-1]
    lap_eigs = np.sort(np.linalg.eigvalsh(L))
    q_eigs = np.sort(np.linalg.eigvalsh(Q))
    norm_lap_eigs = np.sort(np.linalg.eigvalsh(Lnorm))

    spectral_radius = float(np.max(np.abs(adj_eigs)))
    algebraic_connectivity = float(lap_eigs[1]) if n >= 2 else 0.0
    normalized_algebraic_connectivity = float(norm_lap_eigs[1]) if n >= 2 else 0.0
    graph_energy = float(np.sum(np.abs(adj_eigs)))
    avg_degree = 2 * m / n
    laplacian_energy = float(np.sum(np.abs(lap_eigs - avg_degree)))
    estrada_index = float(np.sum(np.exp(adj_eigs)))

    positive_lap_eigs = lap_eigs[lap_eigs > tol]
    kirchhoff_index = float(n * np.sum(1.0 / positive_lap_eigs)) if len(positive_lap_eigs) == n - 1 else np.nan

    L_minor = L[1:, 1:]
    num_spanning_trees = round(np.linalg.det(L_minor))

    pos_inertia = int(np.sum(adj_eigs > tol))
    zero_inertia = int(np.sum(np.abs(adj_eigs) <= tol))
    neg_inertia = int(np.sum(adj_eigs < -tol))

    return {
        "n": n,
        "m": m,
        "spectral_radius": spectral_radius,
        "algebraic_connectivity": algebraic_connectivity,
        "normalized_algebraic_connectivity": normalized_algebraic_connectivity,
        "graph_energy": graph_energy,
        "laplacian_energy": laplacian_energy,
        "estrada_index": estrada_index,
        "kirchhoff_index": kirchhoff_index,
        "num_spanning_trees": num_spanning_trees,
        "adjacency_inertia": (pos_inertia, zero_inertia, neg_inertia),
        "adjacency_eigenvalues": adj_eigs,
        "laplacian_eigenvalues": lap_eigs,
        "signless_laplacian_eigenvalues": q_eigs,
        "normalized_laplacian_eigenvalues": norm_lap_eigs,
    }

In [35]:
test_cave = table_caves.index[0]
print("Testing cave:", test_cave)

G = load_karst_graph(test_cave)
print(G.number_of_nodes(), G.number_of_edges())

inv = spectral_invariants(G)
print(inv["n"], inv["m"], inv["kirchhoff_index"])

Testing cave: GouffreDejaVu
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\001_GouffreDejaVu_000\001_GouffreDejaVu_000.yaml
40 32
Graph was disconnected; using largest connected component.
33 32 5776.000000000034


In [36]:
import yaml

class KarstLoader(yaml.FullLoader):
    pass

def construct_python_tuple(loader, node):
    return tuple(loader.construct_sequence(node))

KarstLoader.add_constructor(
    "tag:yaml.org,2002:python/tuple",
    construct_python_tuple
)

print("KarstLoader is ready")

KarstLoader is ready


In [37]:
def load_karst_graph(cavename):
    id_dataset = table_caves.loc[cavename, "id_dataset"]
    id_subset = table_caves.loc[cavename, "id_subset"]

    yaml_folder = DATA_ROOT / "data" / f"{id_dataset}_{cavename}_{id_subset}"
    path_to_yaml = yaml_folder / f"{id_dataset}_{cavename}_{id_subset}.yaml"

    print("Trying:", path_to_yaml)

    if not path_to_yaml.exists():
        print(f"Missing YAML for {cavename}")
        return None

    with open(path_to_yaml, "r", encoding="utf-8") as f:
        data = yaml.load(f, Loader=KarstLoader)

    G = nx.Graph()

    node_attrs = data.get("node_attributes", {})
    for node, attrs in node_attrs.items():
        if isinstance(attrs, dict):
            clean_attrs = {str(k): v for k, v in attrs.items()}
            G.add_node(node, **clean_attrs)
        else:
            G.add_node(node)

    edges = data.get("edges", [])
    for e in edges:
        if isinstance(e, dict):
            u = e.get("u") or e.get("source")
            v = e.get("v") or e.get("target")
            clean_attrs = {
                str(k): val for k, val in e.items()
                if k not in ["u", "v", "source", "target"]
            }
            if u is not None and v is not None:
                G.add_edge(u, v, **clean_attrs)
        elif isinstance(e, (list, tuple)) and len(e) >= 2:
            G.add_edge(e[0], e[1])

    return G

In [38]:
G = load_karst_graph("Migovec")
print(G.number_of_nodes(), G.number_of_edges())

Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\002_Migovec_000\002_Migovec_000.yaml
7170 7206


In [39]:
results = []

for cavename in table_caves.index:
    try:
        print(f"Processing: {cavename}")

        G = load_karst_graph(cavename)

        if G is None or G.number_of_nodes() == 0:
            print("Skipping empty graph.")
            continue

        inv = spectral_invariants(G)

        row = {
            "cavename": cavename,
            "n": inv["n"],
            "m": inv["m"],
            "spectral_radius": inv["spectral_radius"],
            "algebraic_connectivity": inv["algebraic_connectivity"],
            "normalized_algebraic_connectivity": inv["normalized_algebraic_connectivity"],
            "graph_energy": inv["graph_energy"],
            "laplacian_energy": inv["laplacian_energy"],
            "estrada_index": inv["estrada_index"],
            "kirchhoff_index": inv["kirchhoff_index"],
            "num_spanning_trees": inv["num_spanning_trees"],
        }

        results.append(row)

    except Exception as e:
        print(f"Skipping {cavename}: {e}")

Processing: GouffreDejaVu
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\001_GouffreDejaVu_000\001_GouffreDejaVu_000.yaml
Graph was disconnected; using largest connected component.
Processing: Migovec
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\002_Migovec_000\002_Migovec_000.yaml
Graph was disconnected; using largest connected component.
Processing: Criou
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\003_Criou_000\003_Criou_000.yaml
Graph was disconnected; using largest connected component.
Processing: Matienzo
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\004_Matienzo_000\004_Matienzo_000.yaml
Graph was disconnected; using largest connected component.
Processing: Sakany
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\005_Sakany_000\005_Sakany_000.yaml
Graph was disconnected; using largest connected component.
Processing: ReveEv

C:\Users\roz31\anaconda3\Lib\site-packages\numpy\linalg\_linalg.py:2431: RuntimeWarning: overflow encountered in det
  r = _umath_linalg.det(a, signature=signature)


Skipping Loser: cannot convert float infinity to integer
Processing: Vanoise
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\020_Vanoise_000\020_Vanoise_000.yaml
Graph was disconnected; using largest connected component.
Processing: Glacier
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\021_Glacier_000\021_Glacier_000.yaml
Graph was disconnected; using largest connected component.
Processing: CombeBryon
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\022_CombeBryon_000\022_CombeBryon_000.yaml
Graph was disconnected; using largest connected component.
Processing: Charbonniere
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\023_Charbonniere_000\023_Charbonniere_000.yaml
Graph was disconnected; using largest connected component.
Processing: Lajoux
Trying: C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data\024_Lajoux_000\024_Lajoux_000.yaml
Graph was d

Empty DataFrame
Columns: []
Index: []
Saved karst_spectral_summary.csv


In [40]:
df_results = pd.DataFrame(results)
print(df_results.head())
print("Number of successful caves:", len(df_results))

        cavename     n     m  spectral_radius  algebraic_connectivity  \
0  GouffreDejaVu    33    32         2.084868                0.009304   
1        Migovec  7163  7206         2.712534                0.000002   
2          Criou  3721  3741         2.539595                0.000003   
3       Matienzo  6058  6128         2.944484                0.000003   
4         Sakany  1426  1467         2.591274                0.000254   

   normalized_algebraic_connectivity  graph_energy  laplacian_energy  \
0                       4.939860e-03     41.181433         42.315047   
1                       9.031247e-07   9047.131678       9359.579629   
2                       1.535872e-06   4699.140794       4855.234588   
3                       1.489627e-06   7548.951682       8165.693375   
4                       1.238592e-04   1808.154063       1911.334478   

   estrada_index  kirchhoff_index  \
0      73.966387     5.776000e+03   
1   16531.739882     8.084940e+09   
2    8573.788561 

In [41]:
df_results.to_csv("karst_spectral_summary.csv", index=False)
print("Saved successfully.")

Saved successfully.


In [42]:
print("Processed caves:", len(results))

Processed caves: 77
